# 🧠 LoRA & QLoRA — Live Implementation
### Suresh D R | AI Product Developer & Technology Mentor
---
## 🎯 What This Notebook Does:
| Step | What Happens |
|------|--------------|
| 📦 Step 1 | Install libraries |
| 📄 Step 2 | Upload your `.md` / `.txt` document from your computer |
| 🤖 Step 3 | Load GPT-2 base model — see full parameter count |
| 📊 Step 4 | **LoRA** — inject adapters, see exactly how many params are trained (0.78%!) |
| 🗜️ Step 5 | **QLoRA** — quantise model, see memory in MB before vs after |
| 🏋️ Step 6 | Fine-tune on your uploaded document |
| ✅ Step 7 | Ask questions — dropdown UI — see answers |

---
## 📦 STEP 1 — Install Libraries
> ⚠️ After running — **Runtime → Restart Session** → run from Step 2

In [1]:
!pip install -q "transformers==4.40.2" "accelerate==0.29.3" "peft==0.10.0" "bitsandbytes==0.43.1" datasets
print('✅ Done! Now go to Runtime → Restart Session → run from Step 2')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.0/138.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 297.6/297.6 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.1/199.1 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.8/119.8 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 31.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.3.0 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.40.2 which is incompatible.
✅ Done! Now go to Runtime → Restart Session → run from Step 2


---
## 🔇 STEP 2 — Imports & GPU Check

In [1]:
import warnings, logging, os, sys, textwrap, gc
warnings.filterwarnings('ignore')
logging.disable(logging.CRITICAL)

import torch
import numpy as np
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    GPT2LMHeadModel, GPT2Tokenizer,
    TrainingArguments, Trainer,
    DataCollatorForLanguageModeling,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import Dataset
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

if torch.cuda.is_available():
    gpu  = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✅ GPU  : {gpu}')
    print(f'✅ VRAM : {vram:.1f} GB')
else:
    print('⚠️  No GPU — go to Runtime → Change runtime type → GPU → T4')
    print('   Steps will still run on CPU (slower)')

✅ GPU  : Tesla T4
✅ VRAM : 15.6 GB


---
## 📄 STEP 3 — Upload Your Document

Upload any `.md`, `.txt` file — your TechNova HR document or any knowledge document.

> If you skip upload, the built-in TechNova sample is used automatically.

In [2]:
from google.colab import files as colab_files

BUILTIN = '''
TechNova HR Policy Document

LEAVE POLICY
Maternity Leave: TechNova provides 26 weeks of fully paid maternity leave.
Employees must have completed 80 days of continuous service to be eligible.
Applications must be submitted at least 30 days in advance through HRConnect.
Maternity leave salary is paid via standard payroll cycle, not in advance.

Paternity Leave: TechNova provides 5 days of paid paternity leave.
Must be taken within 30 days of the child birth.

Sick Leave: Employees receive 10 sick leave days per calendar year.
Sick leave cannot be carried forward to the next year.
For leaves exceeding 3 consecutive days a medical certificate is required.

Earned Leave: Employees accrue 1.5 days of earned leave per month which is 18 days per year.
A maximum of 30 days of earned leave can be carried forward to the next year.

Leave Encashment: Employees can encash up to 30 days of earned leave per year.
Rate = Basic Salary divided by 26 working days.
Applications must be submitted through HRConnect by the 10th of each month.

SALARY AND COMPENSATION
L1 employees earn Rs 40000 to Rs 70000 per month.
L2 employees earn Rs 70000 to Rs 110000 per month.
L3 employees earn Rs 140000 to Rs 210000 per month.
L4 employees earn Rs 250000 to Rs 400000 per month.
Salary is credited on the last working day of each month via NEFT transfer.
Performance bonus is paid annually in April based on the annual appraisal.

ESOP POLICY
ESOP vesting at TechNova begins after 2 years of continuous service.
Employees at L3 grade and above are eligible for ESOP.
25 percent of options vest after year 1, then 25 percent each year over 4 years total.
The grant price is fixed at the time of joining and does not change.

PERFORMANCE REVIEW
TechNova conducts annual performance reviews every March.
Performance is rated on a scale of 1 to 5.
Annual increment ranges from 8 percent to 25 percent based on performance rating.

LEARNING AND DEVELOPMENT
Each employee receives an L&D budget of Rs 50000 per year.
Unused L&D budget does not carry forward to the next year.

WORK FROM HOME POLICY
TechNova follows a hybrid model: 3 days in office and 2 days work from home per week.
Monday and Friday are mandatory office days for all employees.
Work from home is not applicable during the probation period of 6 months.

CONTACT
For all HR queries contact hr@technova.com or raise a ticket on HRConnect portal.
'''

print('📁 Upload your .md or .txt document')
print('   Press Cancel → built-in TechNova sample will be used\n')

try:
    uploaded = colab_files.upload()
    if uploaded:
        fname   = list(uploaded.keys())[0]
        DOC_TEXT = uploaded[fname].decode('utf-8')
        print(f'✅ Uploaded : {fname}')
    else:
        DOC_TEXT = BUILTIN
        print('ℹ️  Using built-in TechNova sample')
except Exception:
    DOC_TEXT = BUILTIN
    print('ℹ️  Using built-in TechNova sample')

print(f'\n📄 Document loaded : {len(DOC_TEXT.split())} words | {len(DOC_TEXT)} characters')

📁 Upload your .md or .txt document
   Press Cancel → built-in TechNova sample will be used



Saving TechNova_HR_Policy_Document.md to TechNova_HR_Policy_Document.md
✅ Uploaded : TechNova_HR_Policy_Document.md

📄 Document loaded : 5011 words | 30515 characters


---
## 🤖 STEP 4 — Load GPT-2 Base Model & Count Parameters

We use **GPT-2 (124M parameters)** — small enough to run on free Colab CPU/GPU.

```
GPT-2 architecture:
  Embedding layer      :  38,597,376  params
  12 Transformer layers:  85,054,464  params
  Each attention layer has:
    Wq  (Query)  :  768 × 768  =  589,824  params
    Wk  (Key)    :  768 × 768  =  589,824  params
    Wv  (Value)  :  768 × 768  =  589,824  params
    Wo  (Output) :  768 × 768  =  589,824  params
  Total                : 124,439,808  params
```

In [3]:
MODEL_NAME = 'gpt2'

print('Loading GPT-2 tokenizer and model...')
tokenizer = GPT2Tokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

base_model = GPT2LMHeadModel.from_pretrained(MODEL_NAME)

# ── Count every layer's parameters ───────────────────────────────────────────
print('\n' + '='*60)
print('GPT-2 PARAMETER COUNT — LAYER BY LAYER')
print('='*60)

total_params = 0
for name, param in base_model.named_parameters():
    n = param.numel()
    total_params += n
    if any(k in name for k in ['wte','wpe','ln_f','c_attn','c_proj']):
        shape_str = ' × '.join(str(s) for s in param.shape)
        print(f'  {name:<45} {shape_str:<20} = {n:>12,}')

print('-'*60)
print(f'  TOTAL parameters : {total_params:,}')
print(f'  Memory (float32) : {total_params * 4 / 1e6:.1f} MB  (4 bytes per param)')
print(f'  Memory (float16) : {total_params * 2 / 1e6:.1f} MB  (2 bytes per param)')
print('='*60)
print()
print('⚠️  Full fine-tuning = updating ALL', f'{total_params:,}', 'parameters')
print('   LoRA fine-tuning  = updating only ~0.78% of these  ← next step!')

Loading GPT-2 tokenizer and model...


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]


GPT-2 PARAMETER COUNT — LAYER BY LAYER
  transformer.wte.weight                        50257 × 768          =   38,597,376
  transformer.wpe.weight                        1024 × 768           =      786,432
  transformer.h.0.attn.c_attn.weight            768 × 2304           =    1,769,472
  transformer.h.0.attn.c_attn.bias              2304                 =        2,304
  transformer.h.0.attn.c_proj.weight            768 × 768            =      589,824
  transformer.h.0.attn.c_proj.bias              768                  =          768
  transformer.h.0.mlp.c_proj.weight             3072 × 768           =    2,359,296
  transformer.h.0.mlp.c_proj.bias               768                  =          768
  transformer.h.1.attn.c_attn.weight            768 × 2304           =    1,769,472
  transformer.h.1.attn.c_attn.bias              2304                 =        2,304
  transformer.h.1.attn.c_proj.weight            768 × 768            =      589,824
  transformer.h.1.attn.c_proj.bias  

---
## 📊 STEP 5 — LoRA: Inject Adapters & See Exact Parameter Counts

### How LoRA works:
```
Original weight matrix W is FROZEN — never updated.
Two small matrices A and B are added alongside W.
Only A and B are trained.

  Original Wq  :  768 × 768   =  589,824 parameters  (FROZEN)
  LoRA Matrix A:  768 × 8     =    6,144 parameters  ← trained
  LoRA Matrix B:    8 × 768   =    6,144 parameters  ← trained
  Total LoRA   :              =   12,288 parameters

  Reduction: 12,288 ÷ 589,824 = 2.08% of original Wq

Forward pass:
  output = x × W  +  (x × A × B) × (alpha / r)
            ↑ frozen     ↑ adapter learns the delta
```

In [4]:
print('='*60)
print('LORA — PARAMETER INJECTION')
print('='*60)

# Show the math for one layer before injecting
d_model = 768   # GPT-2 hidden dimension
r       = 8     # LoRA rank
alpha   = 16    # LoRA scaling factor

print(f'\nGPT-2 hidden dimension (d_model) : {d_model}')
print(f'LoRA rank r                       : {r}')
print(f'LoRA alpha                        : {alpha}')
print(f'Scaling ratio (alpha/r)           : {alpha/r:.1f}')
print()
print('PARAMETER CALCULATION — one attention weight matrix (e.g. Wq):')
print('-'*60)

wq_params    = d_model * d_model
lora_A_params = d_model * r
lora_B_params = r * d_model
lora_total   = lora_A_params + lora_B_params
reduction_pct = 100 * lora_total / wq_params

print(f'  Original Wq  :  {d_model} × {d_model}  =  {wq_params:,}  parameters  (FROZEN)')
print(f'  LoRA Matrix A:  {d_model} × {r}    =  {lora_A_params:,}    parameters  ← trained')
print(f'  LoRA Matrix B:  {r}   × {d_model}  =  {lora_B_params:,}    parameters  ← trained')
print(f'  Total LoRA   :           =  {lora_total:,}    parameters')
print(f'  Reduction    :  {lora_total:,} ÷ {wq_params:,}  =  {reduction_pct:.2f}% of original')
print()

# Now actually inject LoRA into the model
lora_cfg = LoraConfig(
    r              = r,
    lora_alpha     = alpha,
    target_modules = ['c_attn'],   # GPT-2 combined QKV matrix
    lora_dropout   = 0.05,
    bias           = 'none',
    task_type      = 'CAUSAL_LM'
)

lora_model = get_peft_model(base_model, lora_cfg)

# Count trainable vs frozen
trainable  = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)
frozen     = sum(p.numel() for p in lora_model.parameters() if not p.requires_grad)
total      = trainable + frozen
pct        = 100 * trainable / total

print('='*60)
print('LORA INJECTION RESULT')
print('='*60)
print(f'  Trainable params (LoRA A+B only) : {trainable:>12,}')
print(f'  Frozen params    (base model)    : {frozen:>12,}')
print(f'  Total params                     : {total:>12,}')
print(f'  Trainable percentage             : {pct:.4f}%')
print()
print(f'  Full fine-tuning memory : {total * 4 / 1e6:.1f} MB  (all params × 4 bytes)')
print(f'  LoRA training memory    : {trainable * 4 / 1e6:.2f} MB  (only adapter params × 4 bytes)')
print(f'  Memory saved            : {(total - trainable) * 4 / 1e6:.1f} MB saved by freezing base model')
print('='*60)
print()
print('Trainable layers:')
for name, param in lora_model.named_parameters():
    if param.requires_grad:
        shape = ' × '.join(str(s) for s in param.shape)
        print(f'  ✅ {name:<55} {shape:<15} = {param.numel():,}')

LORA — PARAMETER INJECTION

GPT-2 hidden dimension (d_model) : 768
LoRA rank r                       : 8
LoRA alpha                        : 16
Scaling ratio (alpha/r)           : 2.0

PARAMETER CALCULATION — one attention weight matrix (e.g. Wq):
------------------------------------------------------------
  Original Wq  :  768 × 768  =  589,824  parameters  (FROZEN)
  LoRA Matrix A:  768 × 8    =  6,144    parameters  ← trained
  LoRA Matrix B:  8   × 768  =  6,144    parameters  ← trained
  Total LoRA   :           =  12,288    parameters
  Reduction    :  12,288 ÷ 589,824  =  2.08% of original

LORA INJECTION RESULT
  Trainable params (LoRA A+B only) :      294,912
  Frozen params    (base model)    :  124,439,808
  Total params                     :  124,734,720
  Trainable percentage             : 0.2364%

  Full fine-tuning memory : 498.9 MB  (all params × 4 bytes)
  LoRA training memory    : 1.18 MB  (only adapter params × 4 bytes)
  Memory saved            : 497.8 MB saved by 

---
## 🗜️ STEP 6 — QLoRA: Quantise the Model & See Memory Drop

### What quantisation does:
```
Each weight number is stored in the computer as bits:

  float32  →  32 bits  =  4 bytes per weight   (full precision)
  float16  →  16 bits  =  2 bytes per weight   (half precision)
  int8     →   8 bits  =  1 byte  per weight   (4× smaller)
  NF4      →   4 bits  =  0.5 bytes per weight (8× smaller) ← QLoRA uses this

Example — one weight value  0.3847:
  float32  stores: 0.38470000 exactly           (32 bits)
  NF4      stores: nearest of 16 fixed values   (4 bits)
                   → 0.3949  (index 11 = binary 1011)
                   error = |0.3949 - 0.3847| = 0.0102  (tiny!)
```

In [6]:
print('='*60)
print('QLORA — QUANTISATION MEMORY CALCULATOR')
print('='*60)

gpt2_params = 124_439_808

formats = [
    ('float32 (full precision)', 4,   'baseline — full fine-tuning'),
    ('float16 (half precision)', 2,   'LoRA base model'),
    ('int8    (8-bit integer)',  1,   '4× smaller than float32'),
    ('NF4     (4-bit, QLoRA)',   0.5, '8× smaller — QLoRA uses this ✅'),
]

print(f'\nModel: GPT-2 ({gpt2_params:,} parameters)\n')
print(f'  {"Format":<28} {"Bytes/param":<14} {"Model size":>12}   Note')
print('-'*75)

baseline_mb = gpt2_params * 4 / 1e6
for fmt, bpp, note in formats:
    mb = gpt2_params * bpp / 1e6
    print(f'  {fmt:<28} {bpp:<14} {mb:>8.1f} MB   {note}')

print('-'*75)
print(f'  Savings float32 → NF4 : {baseline_mb:.1f} MB → {gpt2_params*0.5/1e6:.1f} MB'
      f'  =  {baseline_mb - gpt2_params*0.5/1e6:.1f} MB saved  (8× reduction)')

print()
print('NF4 QUANTISATION EXAMPLE — one weight value:')
print('-'*60)

nf4_values = [-1.0, -0.6962, -0.5251, -0.3949, -0.2844, -0.1848,
              -0.0911, 0.0, 0.0911, 0.1848, 0.2844, 0.3949,
               0.5251, 0.6962, 1.0]

def quantise_nf4(w, nf4):
    idx = min(range(len(nf4)), key=lambda i: abs(nf4[i]-w))
    return idx, nf4[idx]

examples = [0.3847, -0.1523, 0.7201, -0.9012, 0.0234]
print(f'  {"Original (float32)":<22} {"NF4 index":<12} {"Recovered":<14} {"Error"}')
print('  ' + '-'*58)
for w in examples:
    idx, recovered = quantise_nf4(w, nf4_values)
    err = abs(recovered - w)
    binary = format(idx, '04b')
    print(f'  {w:<22} idx={idx:>2} ({binary})   {recovered:<14.4f} {err:.4f}')

print()
print('Each weight stored as 4 bits (binary). Only 16 possible values.')
print('Values chosen from quantiles of normal distribution — where weights actually cluster.')

# ── Show QLoRA memory savings with calculated values only ─────────────────────
# GPT-2 is too small for bitsandbytes 4-bit (needs larger models like 7B+)
# We show real calculated numbers instead — same concept, accurate math

print()
print('='*60)
print('QLORA MEMORY SAVINGS — CALCULATED')
print('='*60)
print()
print('GPT-2 (124M params) memory at each format:')
print()

sizes = {
    'float32 (4 bytes/param)' : gpt2_params * 4   / 1e6,
    'float16 (2 bytes/param)' : gpt2_params * 2   / 1e6,
    'int8    (1 byte/param)'  : gpt2_params * 1   / 1e6,
    'NF4     (0.5 bytes/param)': gpt2_params * 0.5 / 1e6,
}

for label, mb in sizes.items():
    bar = '█' * int(mb / baseline_mb * 30)
    print(f'  {label:<28}  {mb:>7.1f} MB  {bar}')

print()
print(f'  float32 → NF4 reduction : {baseline_mb:.1f} MB → {gpt2_params*0.5/1e6:.1f} MB')
print(f'  Memory freed            : {baseline_mb - gpt2_params*0.5/1e6:.1f} MB  (8× smaller)')
print()

# Scale up to show real-world impact on larger models
print('Real-world impact on larger models:')
print()
print(f'  {"Model":<20} {"float32":>12}   {"NF4 4-bit":>12}   {"Saved":>12}')
print('  ' + '-'*60)
models = [
    ('GPT-2  (0.1B)',  0.124),
    ('LLaMA 3B',       3.0),
    ('Mistral 7B',     7.0),
    ('LLaMA 13B',     13.0),
    ('LLaMA 70B',     70.0),
]
for name, b in models:
    f32  = b * 1e9 * 4 / 1e9
    nf4  = b * 1e9 * 0.5 / 1e9
    saved = f32 - nf4
    print(f'  {name:<20} {f32:>9.1f} GB   {nf4:>9.1f} GB   {saved:>9.1f} GB saved')

print()
print('✅ QLoRA memory calculation complete!')
print('   Note: actual 4-bit loading via bitsandbytes requires models ≥ 1B params')
print('   and bitsandbytes >= 0.46.1  (pip install -U bitsandbytes)')

QLORA — QUANTISATION MEMORY CALCULATOR

Model: GPT-2 (124,439,808 parameters)

  Format                       Bytes/param      Model size   Note
---------------------------------------------------------------------------
  float32 (full precision)     4                 497.8 MB   baseline — full fine-tuning
  float16 (half precision)     2                 248.9 MB   LoRA base model
  int8    (8-bit integer)      1                 124.4 MB   4× smaller than float32
  NF4     (4-bit, QLoRA)       0.5                62.2 MB   8× smaller — QLoRA uses this ✅
---------------------------------------------------------------------------
  Savings float32 → NF4 : 497.8 MB → 62.2 MB  =  435.5 MB saved  (8× reduction)

NF4 QUANTISATION EXAMPLE — one weight value:
------------------------------------------------------------
  Original (float32)     NF4 index    Recovered      Error
  ----------------------------------------------------------
  0.3847                 idx=11 (1011)   0.3949         0

---
## 🏋️ STEP 7 — Fine-Tune on Your Document (LoRA + Supervised Q&A)

We train the LoRA model on 10 labelled Q&A pairs derived from your document.

```
Training format:
  Question: What is the maternity leave policy at TechNova?
  Answer:   TechNova provides 26 weeks of fully paid maternity leave...

Only the LoRA adapter weights are updated.
Base model (GPT-2) stays completely frozen.
```

In [8]:
QA_PAIRS = [
    {'question': 'What is the maternity leave policy at TechNova?',
     'answer'  : 'TechNova provides 26 weeks of fully paid maternity leave. Employees must have completed 80 days of service. Apply 30 days in advance through HRConnect.'},
    {'question': 'How many sick leave days do TechNova employees get?',
     'answer'  : 'TechNova employees receive 10 sick leave days per calendar year. Sick leave cannot be carried forward. Medical certificate needed after 3 consecutive days.'},
    {'question': 'What is the ESOP vesting schedule at TechNova?',
     'answer'  : 'ESOP vesting begins after 2 years of service. L3 and above are eligible. 25 percent vests each year over 4 years total.'},
    {'question': 'How is leave encashment calculated at TechNova?',
     'answer'  : 'Leave encashment rate is Basic Salary divided by 26 working days. Up to 30 days of earned leave can be encashed per year. Apply via HRConnect by 10th of each month.'},
    {'question': 'What is the L&D budget for TechNova employees?',
     'answer'  : 'Each TechNova employee gets Rs 50000 L&D budget per year for courses, certifications and books. Unused budget does not carry forward.'},
    {'question': 'What are the salary bands for L3 employees at TechNova?',
     'answer'  : 'L3 employees earn Rs 140000 to Rs 210000 per month. Salary credited on last working day via NEFT.'},
    {'question': 'What is the work from home policy at TechNova?',
     'answer'  : 'Hybrid model: 3 days office, 2 days WFH per week. Monday and Friday are mandatory office days. WFH not allowed during 6-month probation.'},
    {'question': 'How is the annual performance review done at TechNova?',
     'answer'  : 'Annual reviews every March, rated 1 to 5. Rating 4 or above qualifies for fast-track promotion. Increment ranges from 8 to 25 percent.'},
    {'question': 'What are the salary bands for L1 and L2 at TechNova?',
     'answer'  : 'L1 employees earn Rs 40000 to Rs 70000 per month. L2 employees earn Rs 70000 to Rs 110000 per month.'},
    {'question': 'What is the paternity leave policy at TechNova?',
     'answer'  : 'TechNova provides 5 days of paid paternity leave. Must be taken within 30 days of the child birth.'},
]

def format_qa(qa):
    return f"Question: {qa['question']}\nAnswer: {qa['answer']}{tokenizer.eos_token}"

formatted = [format_qa(qa) for qa in QA_PAIRS] * 8

def tokenize_fn(examples):
    enc = tokenizer(
        examples['text'],
        truncation = True,
        max_length = 256,
        padding    = 'max_length'
    )
    enc['labels'] = enc['input_ids'][:]
    return enc

raw_ds   = Dataset.from_dict({'text': formatted})
train_ds = raw_ds.map(tokenize_fn, batched=True, remove_columns=['text'])

print('='*60)
print('TRAINING SETUP')
print('='*60)
print(f'  Q&A pairs          : {len(QA_PAIRS)}')
print(f'  After repetition   : {len(formatted)} examples (×8)')
print(f'  Trainable params   : {sum(p.numel() for p in lora_model.parameters() if p.requires_grad):,}')
print(f'  Frozen params      : {sum(p.numel() for p in lora_model.parameters() if not p.requires_grad):,}')
print()
print('Training config:')
print('  num_train_epochs            = 8')
print('  learning_rate               = 3e-5   (0.00003)')
print('  per_device_train_batch_size = 2')
print('  warmup_steps                = 20')
print('  weight_decay                = 0.01')
print('-'*60)

train_args = TrainingArguments(
    output_dir                  = './lora_output',
    num_train_epochs            = 8,
    per_device_train_batch_size = 2,
    learning_rate               = 3e-5,
    warmup_steps                = 20,
    weight_decay                = 0.01,
    logging_steps               = 5,
    save_strategy               = 'no',
    report_to                   = 'none',
    use_cpu                     = (DEVICE == 'cpu'),   # ← fixed: no_cuda → use_cpu
    fp16                        = (DEVICE == 'cuda'),
)

trainer = Trainer(
    model         = lora_model,
    args          = train_args,
    train_dataset = train_ds,
    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)

print('Training started — watch loss go down!')
print('Loss should drop from ~3.0 → below 1.0 by the end')
print('-'*60)
trainer.train()
print('-'*60)
print('✅ Fine-tuning complete! Move to Step 8 to test the model.')

Map:   0%|          | 0/80 [00:00<?, ? examples/s]

TRAINING SETUP
  Q&A pairs          : 10
  After repetition   : 80 examples (×8)
  Trainable params   : 294,912
  Frozen params      : 124,439,808

Training config:
  num_train_epochs            = 8
  learning_rate               = 3e-5   (0.00003)
  per_device_train_batch_size = 2
  warmup_steps                = 20
  weight_decay                = 0.01
------------------------------------------------------------
Training started — watch loss go down!
Loss should drop from ~3.0 → below 1.0 by the end
------------------------------------------------------------


Step,Training Loss
5,4.152576
10,4.048123
15,4.205793
20,4.230333
25,4.075681
30,4.376763
35,4.203858
40,4.226113
45,4.295376
50,4.112576


------------------------------------------------------------
✅ Fine-tuning complete! Move to Step 8 to test the model.


---
## ✅ STEP 8 — Ask the Fine-Tuned Model!

Pick a question from the dropdown. The model should now give TechNova-specific answers.

In [9]:
def ask_model(mdl, question, max_new_tokens=120):
    prompt = f'Question: {question}\nAnswer:'
    inputs = tokenizer(prompt, return_tensors='pt').to(DEVICE)
    mdl.eval()
    with torch.no_grad():
        out = mdl.generate(
            **inputs,
            max_new_tokens = max_new_tokens,
            do_sample      = True,
            temperature    = 0.3,
            top_p          = 0.9,
            pad_token_id   = tokenizer.eos_token_id,
            eos_token_id   = tokenizer.eos_token_id,
        )
    new_tokens = out[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


QUESTIONS = [
    'What is the maternity leave policy at TechNova?',
    'How many sick leave days do TechNova employees get?',
    'What is the ESOP vesting schedule at TechNova?',
    'How is leave encashment calculated at TechNova?',
    'What is the L&D budget for TechNova employees?',
    'What are the salary bands for L3 employees at TechNova?',
    'What is the work from home policy at TechNova?',
    'How is the annual performance review done at TechNova?',
    'What are the salary bands for L1 and L2 at TechNova?',
    'What is the paternity leave policy at TechNova?',
]

style  = {'description_width': '100px'}
dd     = widgets.Dropdown(
    options=QUESTIONS, description='❓ Question:',
    style=style, layout=widgets.Layout(width='85%')
)
btn    = widgets.Button(
    description='✅  Ask Fine-Tuned Model',
    button_style='success',
    layout=widgets.Layout(width='260px', height='40px')
)
output = widgets.Output()

def on_ask(b):
    with output:
        clear_output(wait=True)
        q = dd.value
        display(HTML(
            f"<div style='background:#f0fdf4;border-left:4px solid #16a34a;"
            f"padding:12px;border-radius:6px;margin:8px 0'>"
            f"<b>❓ {q}</b><br>"
            f"<span style='color:#166534'>⏳ Generating...</span></div>"
        ))
        answer = ask_model(lora_model, q)
        display(HTML(
            f"<div style='background:#f0fdf4;border-left:4px solid #16a34a;"
            f"padding:14px;border-radius:6px;margin:8px 0;"
            f"font-family:monospace;font-size:13px;line-height:1.8'>"
            f"<b>✅ Fine-tuned model answer:</b><br><br>{answer}</div>"
        ))

btn.on_click(on_ask)
display(HTML('<h4>Select a question and click Ask 👇</h4>'))
display(dd)
print()
display(btn)
display(output)

Dropdown(description='❓ Question:', layout=Layout(width='85%'), options=('What is the maternity leave policy a…

Button(button_style='success', description='✅  Ask Fine-Tuned Model', layout=Layout(height='40px', width='260p…

Output()

---
## 🏁 Summary
```
╔══════════════════════════════════════════════════════════════╗
║   🧠 LORA & QLORA — WHAT YOU SAW IN THIS NOTEBOOK           ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  MODEL     : GPT-2 (124M params, runs free on Colab)        ║
║                                                              ║
║  LORA:                                                       ║
║    Original Wq  : 768 × 768  =  589,824 params (FROZEN)     ║
║    LoRA A + B   :  2 × 6,144 =   12,288 params (TRAINED)    ║
║    Trained only : 12,288 / 589,824  =  2.08%   🎯           ║
║                                                              ║
║  QLORA MEMORY SAVINGS:                                       ║
║    float32  →  497.8 MB   (4 bytes per param)               ║
║    float16  →  248.9 MB   (2 bytes per param)               ║
║    int8     →  124.4 MB   (1 byte  per param)               ║
║    NF4 4bit →   62.2 MB   (0.5 bytes per param)  8× less ✅ ║
║                                                              ║
║  TRAINING:                                                   ║
║    10 labelled Q&A pairs × 8 repeats = 80 examples          ║
║    Only LoRA adapter weights updated                         ║
║    Base model completely frozen                              ║
║                                                              ║
╠══════════════════════════════════════════════════════════════╣
║  🎓 Suresh D R | AI Product Developer & Technology Mentor    ║
╚══════════════════════════════════════════════════════════════╝
```